In [ ]:
"""
Bioactivity Prediction using XGBoost
=====================================
This notebook implements an XGBoost-based pipeline for predicting compound bioactivity.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# RDKit for cheminformatics
import rdkit
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Descriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# Scikit-learn
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef, 
    roc_auc_score, roc_curve, balanced_accuracy_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

# XGBoost and other models
import xgboost as xgb
from xgboost import XGBClassifier


# SHAP for interpretability
import shap

# Scaffold splitting
from splito._scaffold_split import ScaffoldSplit

from deap import base, creator, tools, algorithms

import optuna
from optuna.samplers import TPESampler

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

SEED = 101
TARGET_COL = 'target'
BIO_CLASS = 'class'
DATASET_COL = 'dataset'
NAME_COL = 'molecule_chembl_id'

# Feature columns to exclude
FEATURE_DIFF = ['molecule_chembl_id', 'smiles', 'standard_value', 'class', 
                'pic50', 'scaffold', 'EF', 'scaffold_count']



In [ ]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def filter_features(df, feature_diff, variance_threshold=0.01, correlation_threshold=0.75):
    """
    Filter features by variance and correlation thresholds.
    """
    FEATURES = df.columns.difference(feature_diff).tolist()
    features = df[FEATURES]
    
    # Select numerical features
    numerical_features = features.select_dtypes(include=[np.number])
    
    # Variance filtering
    selector = VarianceThreshold(threshold=variance_threshold)
    numerical_array = selector.fit_transform(numerical_features)
    selected_columns = numerical_features.columns[selector.get_support()].tolist()
    numerical_features = pd.DataFrame(
        numerical_array, 
        index=numerical_features.index, 
        columns=selected_columns
    )
    
    # Correlation filtering
    corr_matrix = numerical_features.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    numerical_features = numerical_features.drop(columns=to_drop)
    
    # Combine with non-numerical features
    non_numerical_features = features.select_dtypes(exclude=[np.number])
    filtered_df = pd.concat([non_numerical_features, numerical_features], axis=1)
    
    return filtered_df


def scale_features(df, scaler=None, fit_scaler=False):
    """
    Scale numerical features using MinMaxScaler.
    """
    numerical_features = df.select_dtypes(include=[np.number])
    
    non_binary_columns = [
        col for col in numerical_features.columns
        if numerical_features[col].nunique() > 2
    ]
    
    if scaler is None:
        scaler = MinMaxScaler()
        fit_scaler = True
    
    if non_binary_columns:
        numerical_features[non_binary_columns] = numerical_features[non_binary_columns].astype('float')
        if fit_scaler:
            scaled_values = scaler.fit_transform(numerical_features[non_binary_columns])
        else:
            scaled_values = scaler.transform(numerical_features[non_binary_columns])
        scaled_df = pd.DataFrame(
            scaled_values, index=numerical_features.index, columns=non_binary_columns
        ).astype('float')
        numerical_features[non_binary_columns] = scaled_df
    
    non_numerical_features = df.select_dtypes(exclude=[np.number])
    result_df = pd.concat([non_numerical_features, numerical_features], axis=1)
    
    return result_df, scaler


def bioactivity_class(value):
    """Convert standard_value to binary class."""
    return 'active' if value <= 7500 else 'inactive'


def y_pred_func(value):
    """Convert probability to binary prediction."""
    return 1 if value >= 0.5 else 0


def perform_xgb_cv(dtrain, params, additional_info=None, seed=SEED):
    """
    Perform cross-validation using XGBoost.
    """
    cv = xgb.cv(
        params=params,
        dtrain=dtrain,
        num_boost_round=5000,
        nfold=5,
        metrics=['auc', 'aucpr', 'logloss'],
        seed=seed,
        early_stopping_rounds=50
    )    
    best_iter = cv['test-aucpr-mean'].idxmax()
    
    result = {
        **params,
        'best_aucpr': cv['test-aucpr-mean'].max(),
        'auc': cv.loc[best_iter, 'test-auc-mean'],
        'best_iteration': best_iter,
        'params': params
    }
    if additional_info:
        result.update(additional_info)
    
    return result


def add_scaffold_enrichment(df):
    """
    Add scaffold information and enrichment factor (EF) for each scaffold.
    """
    # Generate scaffold SMILES
    df['scaffold'] = df['smiles'].apply(
        lambda x: MurckoScaffold.MurckoScaffoldSmiles(smiles=x)
    )
    
    # Calculate enrichment factor
    total_molecules = len(df)
    total_active = len(df[df['class'].isin(['active', 'potent'])])
    p_active_total = total_active / total_molecules
    
    def calc_ef(group):
        n_scaffold = len(group)
        n_active_scaffold = len(group[group['class'].isin(['active', 'potent'])])
        p_active_scaffold = n_active_scaffold / n_scaffold if n_scaffold > 0 else 0
        ef = p_active_scaffold / p_active_total if p_active_total > 0 else 0
        return pd.Series({'EF': ef, 'scaffold_count': n_scaffold})
    
    ef_df = df.groupby('scaffold').apply(calc_ef).reset_index()
    df = df.merge(ef_df, on='scaffold', how='left')
    
    # Filter based on scaffold characteristics
    df = df[~((df['scaffold_count'] > 10) & (df['EF'] < 0.5))]
    df = df[~((df['scaffold_count'] > 5) & (df['EF'] == 0))]
    df = df.reset_index(drop=True)
    
    return df


def evaluate_model_with_cv(model, X, y, cv=5):
    """
    Evaluate model using Stratified Cross Validation.
    """
    scoring = {
        'balanced_accuracy': make_scorer(balanced_accuracy_score),
        'mcc': make_scorer(matthews_corrcoef),
        'precision': make_scorer(precision_score),
        'recall': make_scorer(recall_score),
        'roc_auc': make_scorer(roc_auc_score),
        'f1_score': make_scorer(f1_score)
    }
    
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    
    cv_results = cross_validate(
        model, X, y, 
        cv=skf, 
        scoring=scoring, 
        return_train_score=False,
        n_jobs=-1
    )
    
    results = {}
    for metric in scoring.keys():
        test_scores = cv_results[f'test_{metric}']
        results[metric] = {
            'mean': np.mean(test_scores),
            'std': np.std(test_scores),
            'scores': test_scores
        }
    
    return results


def print_cv_results(results):
    """Print cross-validation results in a formatted way."""
    print("=" * 60)
    print("Stratified Cross Validation Results (CV=5)")
    print("=" * 60)
    
    metrics_names = {
        'balanced_accuracy': 'Balanced Accuracy',
        'mcc': 'Matthews Correlation Coefficient (MCC)',
        'precision': 'Precision',
        'recall': 'Recall',
        'roc_auc': 'AUC-ROC',
        'f1_score': 'F1-Score'
    }
    
    for metric, name in metrics_names.items():
        mean_score = results[metric]['mean']
        std_score = results[metric]['std']
        print(f"{name:35}: {mean_score:.4f} ± {std_score:.4f}")
    
    print("=" * 60)

print("Helper functions defined successfully!")

In [ ]:
# ============================================================================
# LOAD DATA AND ADD BIOACTIVITY CLASS
# ============================================================================

# Load dataset
all_feature = pd.read_csv('../all_feature_train.csv')

# Remove specific molecule
all_feature = all_feature[~(all_feature['smiles'] == DROP_SMILE)]
all_feature.reset_index(drop=True, inplace=True)

# Add bioactivity class
all_feature['class'] = all_feature['standard_value'].apply(bioactivity_class)

print(f"Dataset shape: {all_feature.shape}")
print("\nClass distribution:")
print(all_feature['class'].value_counts())

# ============================================================================
# ADD SCAFFOLD ENRICHMENT
# ============================================================================

all_feature = add_scaffold_enrichment(all_feature)
print(f"After scaffold filtering: {len(all_feature)} molecules")


# ============================================================================
# SCAFFOLD-BASED SPLIT
# ============================================================================

spliter = ScaffoldSplit(
    smiles=all_feature['smiles'].values,
    n_splits=5,
    test_size=0.2,
    random_state=100
)

train_index, test_index = next(spliter.split(all_feature['class']))

all_feature.loc[train_index, 'dataset'] = 'train'
all_feature.loc[test_index, 'dataset'] = 'test'

print("Training data size:", len(train_index))
print("Test data size:", len(test_index))

print("\nClass distribution in TRAIN:")
print(all_feature.loc[train_index, 'class'].value_counts(normalize=True))

print("\nClass distribution in TEST:")
print(all_feature.loc[test_index, 'class'].value_counts(normalize=True))

In [ ]:
# ============================================================================
# FEATURE FILTERING
# ============================================================================

# Update feature_diff to include dataset column
FEATURE_DIFF = [
    'molecule_chembl_id', 'smiles', 'standard_value', 'class', 
    'pic50', 'stage', 'scaffold_count', 'scaffold', 'EF', 'dataset'
]

# Filter features
feature_filtter = filter_features(all_feature, FEATURE_DIFF)
print(f"Features after filtering: {len(feature_filtter.columns)}")

# Create filtered dataset
all_feature_filtered = pd.concat([
    all_feature[['molecule_chembl_id', 'smiles', 'class', 'dataset']], 
    feature_filtter
], axis=1)

print(f"Filtered dataset shape: {all_feature_filtered.shape}")



# ============================================================================
# SCALE FEATURES AND ADD TARGET
# ============================================================================

# Scale features
train_scaled, fitted_scaler = scale_features(
    all_feature_filtered[feature_filtter.columns.to_list()], 
    scaler=None, 
    fit_scaler=True
)

# Create final dataframe with scaled features
df_combine_filter = pd.concat([
    all_feature_filtered[['molecule_chembl_id', 'smiles', 'class', 'dataset']], 
    train_scaled
], axis=1)

# Add target column
df_combine_filter['target'] = df_combine_filter['class'].map({
    'inactive': 0, 
    'active': 1
})

print(f"Final dataset shape: {df_combine_filter.shape}")
print("\nTarget distribution:")
print(df_combine_filter['target'].value_counts())

In [ ]:
# ============================================================================
# HYPERPARAMETER TUNING
# ============================================================================

# Define parameter grid
params = {
    'eta': [0.01, 0.05, 0.1],                    
    'alpha': [0.001, 0.01],                 
    'lambda': [1, 10],                      
    'max_depth': [4, 6],                      
    'subsample': [0.6, 1.0],                
    'colsample_bynode': [0.1, 0.3 ,0.5],         
    'min_child_weight': [2],               
    'gamma': [0.1, 1],                        
    'tree_method': ['hist'],
    'device': ['cpu']
}
params = list(ParameterGrid(params))
print(f"Number of parameter combinations: {len(params)}")

In [ ]:
# ============================================================================
# CROSS-VALIDATION
# ============================================================================

# Get features
FEATURES = df_combine_filter.columns.difference(
    [TARGET_COL, DATASET_COL, NAME_COL, BIO_CLASS, 'target', 'smiles']
).tolist()
print(f"Number of features: {len(FEATURES)}")

# Prepare DMatrix for training
dtrain = xgb.DMatrix(
    df_combine_filter.loc[df_combine_filter['dataset'] == 'train', FEATURES].values,
    label=df_combine_filter.loc[df_combine_filter['dataset'] == 'train', TARGET_COL].values
)

# Perform cross-validation
cv_results = []
for params_id, train_params in enumerate(params):
    result = perform_xgb_cv(
        dtrain, train_params, additional_info={
            'num_features': dtrain.num_col(),
            'n_dtrain': dtrain.num_row(),
            'params_id': params_id
        }
    )
    cv_results.append(result)

# Display results
res = pd.DataFrame(cv_results)
print("\nCross-validation results:")
display(res.sort_values('best_aucpr', ascending=False))

In [ ]:
# ============================================================================
# GET BEST PARAMETERS
# ============================================================================

best_rmse_row = res.loc[res['best_aucpr'].idxmin()]
best_params = best_rmse_row['params'] 
best_iteration = best_rmse_row['best_iteration']

print("Best params:", best_params)
print("Best iteration:", best_iteration)
print("Best AUC-PR:", best_rmse_row['best_aucpr'])

In [ ]:
# ============================================================================
# TRAIN MODEL AND SHAP ANALYSIS
# ============================================================================

# Prepare DMatrix
dtrain = xgb.DMatrix(
    df_combine_filter.loc[df_combine_filter['dataset'] == 'train', FEATURES].values,
    label=df_combine_filter.loc[df_combine_filter['dataset'] == 'train', TARGET_COL].values
)

# Train the model
model = xgb.train(best_params, dtrain, best_iteration)

# SHAP Analysis
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(dtrain)
feature_importance = np.abs(shap_values).mean(axis=0)

shap_values_storage = {
    'shap_values': feature_importance,
    'feature_names': FEATURES
}

# Create SHAP DataFrame
shap_df = pd.DataFrame({
    'feature': shap_values_storage['feature_names'],
    'shap_value': shap_values_storage['shap_values']
})
shap_df = shap_df.sort_values(by='shap_value', ascending=False)

print("Top 10 features by SHAP importance:")
display(shap_df.head(10))

num_zero_shap = len(shap_df[shap_df['shap_value'] == 0])
print(f"Features with SHAP = 0: {num_zero_shap}")

In [ ]:
# ============================================================================
# FEATURE SELECTION BY SHAP PERCENTILES USING GENETIC ALGORITHM
# ============================================================================

import random
import numpy as np


# ----------------------------------------------------------------------
# 1. Define percentile search space
# ----------------------------------------------------------------------
PERCENTILE_OPTIONS = [50, 60, 70, 75, 80, 85, 90, 91, 93, 95, 96, 97]

# ----------------------------------------------------------------------
# 2. Evaluation function
# ----------------------------------------------------------------------
def evaluate_percentile(percentile, df, shap_df, features_all, target_col, best_params, cv_folds=3):
    """
    Evaluate a given percentile threshold using cross-validation.
    Returns mean MCC across CV folds.
    """
    # Select features above the percentile
    shap_values = shap_df['shap_value'].values
    threshold = np.percentile(shap_values, percentile)
    selected_feats = shap_df[shap_df['shap_value'] > threshold]['feature'].tolist()
    
    # Penalty for too few features
    if len(selected_feats) < 5:
        return -1.0
    
    # Prepare data
    X = df.loc[df['dataset'] == 'train', selected_feats].values
    y = df.loc[df['dataset'] == 'train', target_col].values
    
    # Perform cross-validation

    
    model = XGBClassifier(
        n_estimators=500,
        **best_params,
        random_state=42,
        verbosity=0
    )
    
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    mcc_scores = cross_val_score(
        model, X, y, 
        cv=skf, 
        scoring=make_scorer(matthews_corrcoef),
        n_jobs=-1
    )
    
    return np.mean(mcc_scores)

# ----------------------------------------------------------------------
# 3. Setup DEAP genetic algorithm
# ----------------------------------------------------------------------
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Attribute: randomly pick a percentile from the list
toolbox.register("attr_percentile", random.choice, PERCENTILE_OPTIONS)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_percentile, 1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def eval_func(individual):
    percentile = individual[0]
    score = evaluate_percentile(
        percentile, 
        df_combine_filter,  # Your dataframe
        shap_df,            # SHAP importance dataframe
        FEATURES,           # All features
        TARGET_COL,         # Target column
        best_params,        # Best XGBoost parameters
        cv_folds=3
    )
    return (score,)

toolbox.register("evaluate", eval_func)

# Crossover: swap the value between two parents with probability 0.5
def cxUniform(ind1, ind2):
    if random.random() < 0.5:
        ind1[0], ind2[0] = ind2[0], ind1[0]
    return ind1, ind2

# Mutation: change to a different percentile from the list
def mutPercentile(ind):
    current = ind[0]
    options = [p for p in PERCENTILE_OPTIONS if p != current]
    if options:
        ind[0] = random.choice(options)
    return ind,

toolbox.register("mate", cxUniform)
toolbox.register("mutate", mutPercentile)
toolbox.register("select", tools.selTournament, tournsize=3)

# ----------------------------------------------------------------------
# 4. Run the genetic algorithm
# ----------------------------------------------------------------------
print("\n" + "="*60)
print("FEATURE SELECTION USING GENETIC ALGORITHM")
print("="*60)
print(f"Search space: {PERCENTILE_OPTIONS}")
print(f"Population size: 20, Generations: 15")

population_size = 20
num_generations = 15

pop = toolbox.population(n=population_size)
hof = tools.HallOfFame(1)

stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

pop, log = algorithms.eaSimple(
    pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=num_generations,
    stats=stats, halloffame=hof, verbose=True
)

# ----------------------------------------------------------------------
# 5. Extract the best percentile and select features
# ----------------------------------------------------------------------
best_percentile = hof[0][0]
print(f"\n" + "="*60)
print(f"Best percentile found by GA: {best_percentile}")
print("="*60)

# Select features using the best percentile
shap_values = shap_df['shap_value'].values
threshold = np.percentile(shap_values, best_percentile)
FINAL_FEATURES = shap_df[shap_df['shap_value'] > threshold]['feature'].tolist()

print(f"Number of selected features: {len(FINAL_FEATURES)}")
print(f"SHAP threshold: {threshold:.6f}")

# Display top 10 selected features
print("\nTop 10 selected features by SHAP importance:")
print(shap_df[shap_df['feature'].isin(FINAL_FEATURES)].head(10))

# Save the final feature list
with open('best_features_GA.txt', 'w') as f:
    for feat in FINAL_FEATURES:
        f.write(feat + '\n')
print(f"\nFeature list saved to 'best_features_GA.txt'")

# ----------------------------------------------------------------------
# 6. Verify with manual CV on final features
# ----------------------------------------------------------------------
print("\n" + "="*60)
print("VERIFYING FINAL FEATURES WITH CROSS-VALIDATION")
print("="*60)

# Prepare data with final features
X_final = df_combine_filter.loc[
    df_combine_filter['dataset'] == 'train', FINAL_FEATURES
].values
y_final = df_combine_filter.loc[
    df_combine_filter['dataset'] == 'train', TARGET_COL
].values

# Cross-validation on final features
model_final = XGBClassifier(
    n_estimators=500,
    **best_params,
    random_state=42,
    verbosity=0
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
mcc_scores = cross_val_score(
    model_final, X_final, y_final,
    cv=skf,
    scoring=make_scorer(matthews_corrcoef),
    n_jobs=-1
)

print(f"Mean MCC with {len(FINAL_FEATURES)} features: {np.mean(mcc_scores):.4f} ± {np.std(mcc_scores):.4f}")
print("="*60)

In [ ]:
# ----------------------------------------------------------------------
# 1. Prepare data with selected features from GA
# ----------------------------------------------------------------------
# FINAL_FEATURES comes from GA feature selection
dtrain_final = xgb.DMatrix(
    df_combine_filter.loc[df_combine_filter['dataset'] == 'train', FINAL_FEATURES].values,
    label=df_combine_filter.loc[df_combine_filter['dataset'] == 'train', TARGET_COL].values
)

print(f"Training with {len(FINAL_FEATURES)} selected features")
print(f"Training samples: {dtrain_final.num_row()}")

# ----------------------------------------------------------------------
# 2. Define Optuna objective function for XGBoost
# ----------------------------------------------------------------------
def objective_xgb(trial):
    """
    Optuna objective: evaluate XGBoost hyperparameters using CV and return mean AUC-PR.
    """
    params = {
        # Tree structure parameters
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 6),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        
        # Learning parameters
        'eta': trial.suggest_float('eta', 0.001, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.02, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.2, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.1, 0.8),
        
        # Regularization
        'alpha': trial.suggest_float('alpha', 0.0, 1.0, log=True),
        'lambda': trial.suggest_float('lambda', 0.0, 1.0, log=True),
        
        # Fixed parameters
        'tree_method': 'hist',
        'device': 'cpu',
    }
    
    # Perform cross-validation using your perform_xgb_cv function
    result = perform_xgb_cv(
        dtrain_final,
        params,
        additional_info={'trial_number': trial.number}
    )
    
    return result['best_aucpr']

# ----------------------------------------------------------------------
# 3. Run Optuna optimization
# ----------------------------------------------------------------------
print("\n" + "="*60)
print("OPTUNA HYPERPARAMETER OPTIMIZATION")
print("="*60)

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    study_name='xgb_optimization'
)

study.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

# Display best parameters from Optuna
best_params_optuna = study.best_params
best_aucpr_optuna = study.best_value

print(f"\nBest AUC-PR from Optuna: {best_aucpr_optuna:.4f}")
print("Best parameters from Optuna:")
for key, value in best_params_optuna.items():
    print(f"  {key}: {value}")



In [ ]:
# ----------------------------------------------------------------------
# 1. Prepare the final dataset with the features selected from GA
# ----------------------------------------------------------------------
dtrain_final = xgb.DMatrix(
    df_combine_filter.loc[df_combine_filter['dataset'] == 'train', FINAL_FEATURES].values,
    label=df_combine_filter.loc[df_combine_filter['dataset'] == 'train', TARGET_COL].values
)

print(f"Training with {len(FINAL_FEATURES)} selected features")
print(f"Training samples: {dtrain_final.num_row()}")

# ----------------------------------------------------------------------
# 2. Define the refined parameter grid (from Optuna results)
# ----------------------------------------------------------------------
param_grid_final = {
    # Tree structure parameters
    'max_depth': [6, 7, 8],
    'min_child_weight': [2, 3],
    'gamma': [0, 0.5],
    
    # Learning parameters
    'eta': [0.01, 0.03, 0.05],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.1, 0.4, 0.6],
    'colsample_bynode': [0.1, 0.4, 0.6],
    
    # Regularization
    'alpha': [0.1],
    'lambda': [0.1, 10.0],
    
    # Fixed parameters
    'tree_method': ['hist'],
    'device': ['cpu'],
}

# Generate all combinations
final_params_list = list(ParameterGrid(param_grid_final))
print(f"Total parameter combinations: {len(final_params_list)}")

# ----------------------------------------------------------------------
# 3. Evaluate each combination with cross-validation
# ----------------------------------------------------------------------
final_cv_results = []

for i, params in enumerate(final_params_list):
    result = perform_xgb_cv(
        dtrain_final, 
        params,
        additional_info={
            'num_features': dtrain_final.num_col(),
            'n_dtrain': dtrain_final.num_row(),
            'params_id': i
        }
    )
    final_cv_results.append(result)
    
    # Print progress
    if (i + 1) % 5 == 0 or i == len(final_params_list) - 1:
        print(f"Processed {i+1}/{len(final_params_list)} combinations")

# ----------------------------------------------------------------------
# 4. Find the best combination
# ----------------------------------------------------------------------
final_cv_df = pd.DataFrame(final_cv_results)
best_final_row = final_cv_df.loc[final_cv_df['best_aucpr'].idxmax()]

print("\n" + "="*60)
print("FINAL BEST PARAMETERS (from manual grid search)")
print("="*60)
print(f"Best AUC-PR: {best_final_row['best_aucpr']:.4f}")
print(f"Best iteration: {best_final_row['best_iteration']}")
print("Best parameters:")
for key, value in best_final_row['params'].items():
    print(f"  {key}: {value}")

FINAL_PARAMS = best_final_row['params']
FINAL_ITER = int(best_final_row['best_iteration'])

# ----------------------------------------------------------------------
# 5. Train final model with best parameters
# ----------------------------------------------------------------------
print("\n" + "="*60)
print("TRAINING FINAL MODEL")
print("="*60)

# Train final model
model_final = xgb.train(
    FINAL_PARAMS,
    dtrain_final,
    num_boost_round=FINAL_ITER
)

# Evaluate on test set
test_indices = df_combine_filter.loc[df_combine_filter['dataset'] == 'test'].index.tolist()
dtest_final = xgb.DMatrix(df_combine_filter.loc[test_indices, FINAL_FEATURES].values)
predictions = model_final.predict(dtest_final)

# Calculate metrics
from sklearn.metrics import matthews_corrcoef, accuracy_score, f1_score, precision_score, recall_score

y_true = df_combine_filter.loc[test_indices, TARGET_COL].values.astype(int)
y_pred = (predictions >= 0.5).astype(int)

print(f"Test MCC: {matthews_corrcoef(y_true, y_pred):.4f}")
print(f"Test Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Test F1-Score: {f1_score(y_true, y_pred):.4f}")
print("="*60)

# ----------------------------------------------------------------------
# 6. Save results
# ----------------------------------------------------------------------
# Save best parameters
with open('best_xgb_params_final.txt', 'w') as f:
    f.write(f"Best AUC-PR: {best_final_row['best_aucpr']:.4f}\n")
    f.write(f"Best iteration: {FINAL_ITER}\n")
    f.write(f"Number of features: {len(FINAL_FEATURES)}\n")
    for key, value in FINAL_PARAMS.items():
        f.write(f"{key}: {value}\n")

print("\nBest parameters saved to 'best_xgb_params_final.txt'")

## Train Model

In [ ]:
# ============================================================================
# LOAD DATA
# ============================================================================

# Load feature list from file
with open('output97-145.txt', 'r', encoding='utf-8') as f:
    my_list = [line.strip() for line in f]

print(f"Loaded {len(my_list)} features from file")

# Load datasets
all_feature = pd.read_csv('../all_feature_train.csv')
df = pd.read_csv("df_fillter_dataset-with-xgboost.csv")

# Remove specific molecule
all_feature.reset_index(drop=True, inplace=True)

# Create main dataframe with selected features
df_main = pd.concat([
    all_feature[['molecule_chembl_id', 'smiles', 'standard_value', 'class', 'pic50']], 
    all_feature[my_list]
], axis=1)

print(f"All feature shape: {all_feature.shape}")
print(f"Main dataframe shape: {df_main.shape}")
print(f"df shape: {df.shape}")

In [ ]:
# ============================================================================
# ADD BIOACTIVITY CLASS AND SCAFFOLD INFO
# ============================================================================

# Add bioactivity class
df_main['class'] = df_main['standard_value'].apply(bioactivity_class)
print("Class distribution in df_main:")
print(df_main['class'].value_counts())

# Rename and prepare herbal dataset
df = df.rename(columns={"name": "molecule_chembl_id"})
df['class'] = ['unknown' for _ in range(len(df))]

# Add scaffold enrichment
df_main = add_scaffold_enrichment(df_main)
print(f"\nAfter scaffold filtering: {len(df_main)} molecules")

In [ ]:
# ============================================================================
# SCAFFOLD-BASED SPLIT
# ============================================================================

spliter = ScaffoldSplit(
    smiles=df_main['smiles'].values,
    n_splits=5,
    test_size=0.2,
    random_state=100
)

train_index, test_index = next(spliter.split(df_main['class']))

df_main.loc[train_index, 'dataset'] = 'train'
df_main.loc[test_index, 'dataset'] = 'test'

print("Training data size:", len(train_index))
print("Test data size:", len(test_index))

print("\nClass distribution in TRAIN:")
print(df_main.loc[train_index, 'class'].value_counts(normalize=True))

print("\nClass distribution in TEST:")
print(df_main.loc[test_index, 'class'].value_counts(normalize=True))

In [ ]:
# ============================================================================
# FEATURE FILTERING AND PREPROCESSING
# ============================================================================

# Update feature_diff to include dataset column
FEATURE_DIFF = ['molecule_chembl_id', 'smiles', 'standard_value', 'class', 
                'pic50', 'stage', 'scaffold_count', 'scaffold', 'EF', 'dataset']

# Filter features
feature_filtter = filter_features(df_main, FEATURE_DIFF)

# Create filtered dataset
all_feature_filtered = pd.concat([
    df_main[['molecule_chembl_id', 'smiles', 'class', 'dataset']], 
    feature_filtter
], axis=1)

# Filter herbal dataset
df_herbal_filtered = df[all_feature_filtered.columns.to_list()]
df_herbal_filtered = df_herbal_filtered.dropna(axis=0)

# Combine datasets
df_combine = pd.concat([all_feature_filtered, df_herbal_filtered], axis=0)
df_combine.reset_index(drop=True, inplace=True)

print(f"Combined dataset shape: {df_combine.shape}")
print(f"Dataset distribution:\n{df_combine['dataset'].value_counts()}")

# ============================================================================
# LOG TRANSFORMATION FOR NON-BINARY FEATURES
# ============================================================================


# Separate training and screening data
train_df = all_feature_filtered[my_list]  # Training data
screen_df = df_herbal_filtered[my_list]   # Screening compounds

# Identify non-binary columns
non_binary_columns = [
    col for col in train_df.columns
    if train_df[col].nunique() > 2 
]

# Columns requiring special handling
neg_cols = [col for col in non_binary_columns if train_df[col].min() < 0]
non_neg_cols = list(set(non_binary_columns) - set(neg_cols))

# Custom transformers
def safe_log1p(X):
    """Handles zeros with pseudo-count"""
    X = X.copy()
    zeros_mask = X <= 0
    X[zeros_mask] = 1e-8
    return np.log1p(X)

def shifted_log(X):
    """Handles negative values"""
    shift = np.abs(X.min()) + 1e-8
    return np.log1p(X + shift)

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('shifted_log', FunctionTransformer(shifted_log), neg_cols),
        ('safe_log', FunctionTransformer(safe_log1p), non_neg_cols),
        ('passthrough', 'passthrough', list(set(train_df.columns) - set(non_binary_columns)))
    ],
    remainder='drop'
)

# Transform data
X_train_transformed = preprocessor.fit_transform(train_df)
X_screen_transformed = preprocessor.transform(screen_df)

# Convert to DataFrames
transformed_columns = (
    neg_cols + 
    non_neg_cols + 
    list(set(train_df.columns) - set(non_binary_columns))
)

X_train_log = pd.DataFrame(
    X_train_transformed, 
    columns=transformed_columns, 
    index=train_df.index
)

X_screen_log = pd.DataFrame(
    X_screen_transformed, 
    columns=transformed_columns, 
    index=screen_df.index
)

# Combine log-transformed data
df_combine_log = pd.concat([X_train_log, X_screen_log], axis=0)
df_combine_log.reset_index(drop=True, inplace=True)

# Final combined dataframe with log-transformed features
df_combine_filter = pd.concat([
    df_combine[['molecule_chembl_id', 'smiles', 'class', 'dataset']], 
    df_combine_log
], axis=1)

print(f"Final combined shape: {df_combine_filter.shape}")

In [ ]:
# ============================================================================
# ADD TARGET COLUMN
# ============================================================================

# Map class to target values
df_combine_filter['target'] = df_combine_filter['class'].map({
    'inactive': 0, 
    'active': 1, 
    'unknown': 'unknown'
})

print("Target distribution:")
print(df_combine_filter['target'].value_counts())

In [ ]:
# ============================================================================
# HYPERPARAMETER TUNING
# ============================================================================

TARGET_COL = 'target'
BIO_CLASS = 'class'
DATASET_COL = 'dataset'
NAME_COL = 'molecule_chembl_id'

# Define parameter grid
params = {
    'eta': [0.01],
    'alpha': [0.01],
    'lambda': [10],
    'max_depth': [6],
    'min_child_weight': [2],
    'gamma': [0],
    'subsample': [0.6],
    'colsample_bytree': [0.1],
    'colsample_bynode': [0.6],
    'scale_pos_weight': [2],
    'tree_method': ['hist'],
}

params = list(ParameterGrid(params))
print(f"Number of parameter combinations: {len(params)}")

In [ ]:
# ============================================================================
# CROSS-VALIDATION
# ============================================================================

# Get features (exclude target and identifier columns)
FEATURES = df_combine_filter.columns.difference(
    [TARGET_COL, DATASET_COL, NAME_COL, BIO_CLASS, 'target', 'smiles']
).tolist()
print(f"Number of features: {len(FEATURES)}")

# Prepare DMatrix for training
dtrain = xgb.DMatrix(
    df_combine_filter.loc[df_combine_filter['dataset'] == 'train', FEATURES].values,
    label=df_combine_filter.loc[df_combine_filter['dataset'] == 'train', TARGET_COL].values
)

# Perform cross-validation
cv_results = []
for params_id, train_params in enumerate(params):
    result = perform_xgb_cv(
        dtrain, train_params, additional_info={
            'num_features': dtrain.num_col(),
            'n_dtrain': dtrain.num_row(),
            'params_id': params_id
        }
    )
    cv_results.append(result)

# Display results
cv_df = pd.DataFrame(cv_results)
print("\nCross-validation results:")
display(cv_df.sort_values('best_aucpr', ascending=False))

In [ ]:
# ============================================================================
# GET BEST PARAMETERS
# ============================================================================

# Aggregate results by params_id
best_params_df = pd.DataFrame(cv_results).groupby('params_id').agg(
    {'best_aucpr': 'mean', 'best_iteration': 'mean', 'params': 'first'}
)

# Find best parameters
best_row = best_params_df.loc[best_params_df['best_aucpr'].idxmax()]

best_params = best_row['params']
best_iteration = int(best_row['best_iteration'])
best_aucpr = best_row['best_aucpr']

print("Best params:", best_params)
print("Best iteration:", best_iteration)
print("Best AUC-PR:", best_aucpr)

In [ ]:
# ============================================================================
# TRAIN AND EVALUATE ON TEST SET
# ============================================================================

# Train final model
bst = xgb.train(
    best_params,
    dtrain,
    num_boost_round=best_iteration
)

# Prepare test data
test_indices = df_combine_filter.loc[df_combine_filter['dataset'] == 'test'].index.tolist()
dtest = xgb.DMatrix(df_combine_filter.loc[test_indices, FEATURES].values)
predictions = bst.predict(dtest)

# Create submission dataframe
submit = pd.DataFrame({
    'Index': test_indices,
    'Predictions': predictions
})
submit['y_pred'] = submit['Predictions'].apply(y_pred_func)

# Get true labels
true_labels = df_combine_filter.loc[test_indices, 'target'].values.astype(int)
y_pred = submit['y_pred'].values

# Calculate metrics
cm = confusion_matrix(true_labels, y_pred)
mcc = matthews_corrcoef(true_labels, y_pred)
accuracy = accuracy_score(true_labels, y_pred)
precision = precision_score(true_labels, y_pred)
recall = recall_score(true_labels, y_pred)
f1 = f1_score(true_labels, y_pred)

print("Confusion Matrix:")
print(cm)
print("\n" + "="*50)
print("TEST SET PERFORMANCE")
print("="*50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"MCC:       {mcc:.4f}")
print("="*50)

In [ ]:
# ============================================================================
# PREDICT ON HERBAL COMPOUNDS
# ============================================================================

# Train on all non-herbal data
dtrain_all = xgb.DMatrix(
    df_combine_filter.loc[df_combine_filter['dataset'] != 'herbal', FEATURES].values,
    label=df_combine_filter.loc[df_combine_filter['dataset'] != 'herbal', TARGET_COL].values
)

bst_all = xgb.train(
    best_params,
    dtrain_all,
    num_boost_round=best_iteration
)

# Predict on herbal compounds
herbal_indices = df_combine_filter.loc[df_combine_filter['dataset'] == 'herbal'].index.tolist()
dtest_herbal = xgb.DMatrix(df_combine_filter.loc[herbal_indices, FEATURES].values)
herbal_predictions = bst_all.predict(dtest_herbal)

# Create submission for herbal compounds
submit_herbal = pd.DataFrame({
    'Index': herbal_indices,
    'Predictions': herbal_predictions
})
submit_herbal['y_pred'] = submit_herbal['Predictions'].apply(y_pred_func)

print(f"Herbal compounds predicted: {len(submit_herbal)}")
display(submit_herbal.head(10))